# Smart Irrigation Water Demand Predictor - Exploratory Data Analysis

This notebook performs exploratory data analysis on the irrigation dataset, including data loading, preprocessing, visualization, and model training.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import joblib
import os

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Load and Explore the Dataset

In [ ]:
# Load the dataset
data_path = '../../Crop_recommendation.csv'
if os.path.exists(data_path):
    data = pd.read_csv(data_path)
    print("Crop_recommendation.csv loaded successfully!")
    
    # Add water_required column
    data['water_required'] = (
        data['temperature'] * 0.5 +
        (100 - data['humidity']) * 0.3 -
        data['rainfall'] * 0.01 +
        (data['N'] + data['P'] + data['K']) * 0.001 +
        np.random.normal(0, 5, len(data))  # Add some noise
    )
    data['water_required'] = np.maximum(data['water_required'], 0)
    
else:
    print("Dataset not found. Please ensure Crop_recommendation.csv exists in the parent directory.")
    # For demonstration, generate sample data
    np.random.seed(42)
    data = pd.DataFrame({
        'temperature': np.random.uniform(15, 35, 100),
        'humidity': np.random.uniform(40, 90, 100),
        'rainfall': np.random.uniform(0, 300, 100),
        'N': np.random.uniform(0, 140, 100),
        'P': np.random.uniform(0, 145, 100),
        'K': np.random.uniform(0, 205, 100),
        'water_required': np.random.uniform(0, 50, 100)
    })

# Display basic information
print(f"Dataset shape: {data.shape}")
print("\nFirst 5 rows:")
display(data.head())

print("\nDataset info:")
data.info()

print("\nBasic statistics:")
display(data.describe())

## 2. Data Preprocessing and Missing Values Check

In [ ]:
# Check for missing values
print("Missing values in each column:")
missing_values = data.isnull().sum()
print(missing_values)

# Percentage of missing values
print("\nPercentage of missing values:")
print((missing_values / len(data)) * 100)

# Handle missing values (if any)
if data.isnull().sum().sum() > 0:
    print("\nHandling missing values...")
    # For numerical columns, fill with mean
    numerical_cols = data.select_dtypes(include=[np.number]).columns
    data[numerical_cols] = data[numerical_cols].fillna(data[numerical_cols].mean())
    print("Missing values filled with column means.")
else:
    print("\nNo missing values found.")

# Check for duplicates
print(f"\nNumber of duplicate rows: {data.duplicated().sum()}")

# Remove duplicates if any
if data.duplicated().sum() > 0:
    data = data.drop_duplicates()
    print("Duplicates removed.")

print(f"\nFinal dataset shape: {data.shape}")

## 3. Create Target Variable: water_required

In [ ]:
# Check if target variable exists
if 'water_required' not in data.columns:
    print("Creating target variable 'water_required'...")
    # Formula based on environmental conditions
    data['water_required'] = (
        data['temperature'] * 0.5 +
        (100 - data['humidity']) * 0.3 -
        data['rainfall'] * 0.01 +
        (data['N'] + data['P'] + data['K']) * 0.001 +
        np.random.normal(0, 5, len(data))  # Add some noise
    )
    # Ensure non-negative values
    data['water_required'] = np.maximum(data['water_required'], 0)
    print("Target variable created.")
else:
    print("Target variable 'water_required' already exists.")

print(f"Target variable statistics:")
print(data['water_required'].describe())

## 4. Exploratory Data Analysis (EDA) with Visualizations

In [ ]:
# Distribution of features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Distribution of Features', fontsize=16)

features = ['temperature', 'humidity', 'rainfall', 'N', 'P', 'K']
for i, feature in enumerate(features):
    row = i // 3
    col = i % 3
    axes[row, col].hist(data[feature], bins=30, edgecolor='black')
    axes[row, col].set_title(f'Distribution of {feature}')
    axes[row, col].set_xlabel(feature)
    axes[row, col].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Distribution of target variable
plt.figure(figsize=(8, 6))
plt.hist(data['water_required'], bins=30, edgecolor='black')
plt.title('Distribution of Water Required')
plt.xlabel('Water Required')
plt.ylabel('Frequency')
plt.show()

# Correlation heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

# Scatter plots of features vs target
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Features vs Water Required', fontsize=16)

for i, feature in enumerate(features):
    row = i // 3
    col = i % 3
    axes[row, col].scatter(data[feature], data['water_required'], alpha=0.5)
    axes[row, col].set_xlabel(feature)
    axes[row, col].set_ylabel('Water Required')
    axes[row, col].set_title(f'{feature} vs Water Required')

plt.tight_layout()
plt.show()

# Box plots for outlier detection
plt.figure(figsize=(12, 6))
data[features + ['water_required']].boxplot()
plt.title('Box Plots of All Variables')
plt.xticks(rotation=45)
plt.show()

## 5. Split Dataset into Training and Testing Sets

In [ ]:
# Define features and target
X = data[features]
y = data['water_required']

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")

# Display sample of training data
print("\nSample of training features:")
display(X_train.head())
print("\nSample of training target:")
display(y_train.head())

## 6. Train Regression Model using RandomForestRegressor

In [ ]:
# Initialize the model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    max_depth=10
)

# Train the model
model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"Model parameters: {model.get_params()}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
display(feature_importance)

# Plot feature importance
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feature_importance)
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

## 7. Evaluate Model with R2 Score and Mean Squared Error

In [ ]:
# Make predictions on test set
y_pred = model.predict(X_test)

# Calculate evaluation metrics
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("Model Evaluation Metrics:")
print(f"R² Score: {r2:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

# Plot actual vs predicted
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Water Required')
plt.ylabel('Predicted Water Required')
plt.title('Actual vs Predicted Water Required')
plt.grid(True)
plt.show()

# Residual plot
residuals = y_test - y_pred
plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Water Required')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.grid(True)
plt.show()

# Distribution of residuals
plt.figure(figsize=(8, 6))
plt.hist(residuals, bins=30, edgecolor='black')
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Distribution of Residuals')
plt.show()

## 8. Save Trained Model using joblib

In [ ]:
# Save the trained model
model_path = '../models/irrigation_model.pkl'
os.makedirs(os.path.dirname(model_path), exist_ok=True)
joblib.dump(model, model_path)

print(f"Model saved successfully to {model_path}")
print(f"Model file size: {os.path.getsize(model_path)} bytes")

# Verify the saved model
loaded_model = joblib.load(model_path)
print("Model loaded successfully for verification.")

# Test the loaded model with a sample prediction
sample_prediction = loaded_model.predict(X_test.iloc[:1])
print(f"Sample prediction with loaded model: {sample_prediction[0]:.2f}")

## 9. Create Prediction Script

In [ ]:
# Demonstration of prediction functionality
def predict_water_requirement(input_features):
    """
    Predict irrigation water requirement for given features.

    Parameters:
    input_features (dict): Dictionary of feature values

    Returns:
    float: Predicted water requirement
    """
    # Convert input to numpy array
    features_array = np.array([[
        input_features['temperature'],
        input_features['humidity'],
        input_features['rainfall'],
        input_features['N'],
        input_features['P'],
        input_features['K']
    ]])

    # Make prediction
    prediction = model.predict(features_array)[0]
    return prediction

# Example predictions
print("Prediction Examples:")
examples = [
    {'temperature': 25.0, 'humidity': 60.0, 'rainfall': 50.0, 'N': 80.0, 'P': 40.0, 'K': 40.0},
    {'temperature': 30.0, 'humidity': 50.0, 'rainfall': 20.0, 'N': 90.0, 'P': 50.0, 'K': 50.0},
    {'temperature': 20.0, 'humidity': 80.0, 'rainfall': 100.0, 'N': 60.0, 'P': 30.0, 'K': 30.0}
]

for i, example in enumerate(examples, 1):
    pred = predict_water_requirement(example)
    print(f"Example {i}: Predicted water requirement = {pred:.2f} units")
    print(f"  Input: {example}")
    print()

## 10. Flask Web Application for Predictions

The Flask web application code is in `app/app.py`. Here's a summary of how it works:

1. Loads the trained model
2. Provides a web form for inputting environmental features
3. Makes predictions and displays results
4. Runs on `http://127.0.0.1:5000/`

To run the app:
```python
python app/app.py
```

The application includes:
- HTML form for user input
- Server-side prediction using the trained model
- Results display
- Error handling